In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import os
import sys
import tensorflow as tf
import STAGATE

In [ ]:
# This notebook performs STAGATE-based spatial domain clustering on bin100 Stereo-seq data. 
# To identify subclusters within the regression region (ROI_RR), first extract the ROI_RR spots 
# from the initial clustering result, then rerun this notebook using that subset as input.

In [ ]:
def run_STAGATE(sample_id, binsize, input_path, n_top, cutoff, output_path):
    adata = sc.read_10x_mtx(input_path, var_names='gene_symbols', cache=True)

    xy_pos=pd.read_csv(f'{input_path}/xy_pos.csv',index_col=0)
    xy_pos = xy_pos.drop_duplicates(subset=['x', 'y'])
    coor_df = xy_pos.loc[adata.obs_names, ['y', 'x']]

    adata.obsm["spatial"] = coor_df.to_numpy()
    sc.pp.calculate_qc_metrics(adata, inplace=True)
    plt.rcParams["figure.figsize"] = (5,4)
    sc.pl.embedding(adata, basis="spatial", color="n_genes_by_counts", show=False)
    plt.title("")
    plt.axis('off')

    adata.layers['counts'] = adata.X.copy()

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers['normalize'] = adata.X.copy()

    sc.pp.highly_variable_genes(adata,n_top_genes = n_top)
    adata_hvg=adata[:, adata.var['highly_variable']].copy()

    STAGATE.Cal_Spatial_Net(adata_hvg, rad_cutoff = cutoff)
    STAGATE.Stats_Spatial_Net(adata_hvg)

    tf.compat.v1.disable_eager_execution()
    adata_hvg = STAGATE.train_STAGATE(adata_hvg, alpha=0)

    sc.pp.neighbors(adata_hvg, use_rep='STAGATE')
    sc.tl.umap(adata_hvg)

    adata_hvg.write_h5ad(f'{output_path}/{sample_id}_bin{binsize}_hvg.h5ad')
    adata.write_h5ad(f'{output_path}/{sample_id}_bin{binsize}.h5ad')

In [ ]:
## DEG

In [ ]:
sample_id = 'sample_tmp'
binsize = 100
input_path = '/data/work/STAGATE/result'

In [ ]:
adata_hvg = sc.read_h5ad(f'{input_path}/{sample_id}_bin{binsize}_hvg.h5ad')
adata = sc.read_h5ad(f'{input_path}/{sample_id}_bin{binsize}.h5ad')

In [ ]:
numlist=np.arange(0.1,2.5,0.1)
res_list=[]
for num in numlist:
    num=round(num,1)
    if num.is_integer():
        num = int(num)
    res_list.append(num)
print(res_list)

In [ ]:
successful_resolutions = []
for res in res_list:
    try:
        res_path=f'{input_path}/res{res}'
        os.makedirs(res_path, exist_ok=True)
        print(res_path)
        sc.tl.louvain(adata_hvg, resolution=res)
        res_name=f'louvain_res{res}'
        adata_hvg.obs[res_name]=adata_hvg.obs['louvain']
        adata.obs[res_name]=adata_hvg.obs['louvain']
        
        plt.rcParams["figure.figsize"] = (3, 3)
        sc.pl.embedding(adata_hvg, basis="spatial", color="louvain",s=2, show=False, title='STAGATE')
        plt.axis('off')
        plt.savefig(f'{res_path}/embedding_plot_res{res}.png', format='png', dpi=300, bbox_inches='tight')
        plt.close()
    
        plt.rcParams["figure.figsize"] = (3, 3)
        sc.pl.umap(adata_hvg, color='louvain', title='STAGATE',show=False)
        plt.savefig(f'{res_path}/umap_plot_res{res}.png', format='png', dpi=300, bbox_inches='tight')
        plt.close()
    
        sc.tl.rank_genes_groups(adata, res_name, method="wilcoxon")
        groups=list(adata.obs[res_name].unique())
        dfs = []
        for group in groups:
            df = sc.get.rank_genes_groups_df(adata, group=group)
            df['cluster'] = group
            dfs.append(df)
        df_all = pd.concat(dfs)
        df_all = df_all.sort_values(by="logfoldchanges", ascending=False)
        df_all.to_csv(f'{res_path}/{sample_id}_bin{binsize}_res{res}.csv',index=False)
        print(f'{res_path}/{sample_id}_bin{binsize}_res{res}.csv')
        successful_resolutions.append(res)
    except Exception as e:
        print(f"Error with resolution {res}: {e}")
        continue

In [ ]:
adata_hvg.write_h5ad(f'{input_path}/{sample_id}_bin{binsize}_hvg.h5ad')
adata.write_h5ad(f'{input_path}/{sample_id}_bin{binsize}.h5ad')

In [ ]:
for res in successful_resolutions:
    res_path=f'{input_path}/res{res}'
    res_csv=pd.read_csv(f'{res_path}/{sample_id}_bin{binsize}_res{res}.csv')
    res_csv=res_csv.loc[(res_csv['pvals_adj']<0.05) & (res_csv['logfoldchanges']>0.25)]
    res_csv = res_csv.sort_values(by='logfoldchanges', ascending=False)
    zhang_M_anno=pd.read_csv('/data/work/file/zhang_M.csv')
    zhang_M_anno.rename(columns={'ID':'names'},inplace=True)
    genes_zhang_M_anno=pd.merge(res_csv,zhang_M_anno,on='names',how='left')
    genes_zhang_M_anno.to_csv(f'{res_path}/{sample_id}_bin{binsize}_res{res}_zhang_M.csv',index=False)
    cluster_list=list(genes_zhang_M_anno.loc[:,'cluster'].unique())
    cluster_path=f'{res_path}/{sample_id}_bin{binsize}_res{res}_zhang_M'
    os.makedirs(cluster_path, exist_ok=True)
    print(cluster_path)
    for cluster in cluster_list:
        cluster_csv=genes_zhang_M_anno[genes_zhang_M_anno['cluster']==cluster]
        cluster_csv = cluster_csv.sort_values(by="logfoldchanges", ascending=False)
        cluster_csv.to_csv(f'{cluster_path}/{cluster}.csv',index=False)